In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

plant_pathology_2020_fgvc7_path = kagglehub.competition_download('plant-pathology-2020-fgvc7')

print('Data source import complete.')


# 시드값 고정 및 GPU 장비 설정

## 시드값 고정

In [ ]:
import torch
import random
import numpy as np
import os

In [ ]:
seed = 50 # 시드값 고정
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.beenchmark = False
torch.backends.cudnn.enabled = False

## GPU 장비 설정

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

# 데이터 준비

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data_path = '/kaggle/input/plant-pathology-2020-fgvc7/'

In [ ]:
train = pd.read_csv(data_path + 'train.csv')
test = pd.read_csv(data_path + 'test.csv')
submission = pd.read_csv(data_path + 'sample_submission.csv')

## 훈련 데이터, 검증 데이터 분리

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train, valid = train_test_split(train,
                                test_size=0.1,
                                stratify=train[['healthy', 'multiple_diseases', 'rust', 'scab']],
                                random_state=50)

## 데이터셋 클래스 정의

In [ ]:
import cv2
from torch.utils.data import Dataset

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, df, img_dir='./', transform=None, is_test=False):
        super().__init__()
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx, 0]
        img_path = self.img_dir + img_id + '.jpg'
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)['image']

        # 테스트 데이터면 이미지 데이터만 반환, 그렇지 않으면 타깃값도 반환
        if self.is_test:
            return image # 테스트용일 때
        else:
            label = np.argmax(self.df.iloc[idx, 1:5])
            return image, label # 훈련/검증용일 때

        return image, label

## 이미지 변환기 정의

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
transform_train = A.Compose([
    A.Resize(450, 650),                                     # 이미지 크기 조절
    A.RandomBrightnessContrast(brightness_limit=0.2,        # 밝기 대비 조절
                               contrast_limit=0.2, p=0.3),
    A.VerticalFlip(p=0.2),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(                                     # 이동, 스케일링, 회전 변환
        shift_limit=0.1,
        scale_limit=0.2,
        rotate_limit=30, p=0.3),
    A.OneOf([A.Emboss(p=1),                                 # 양각화, 날카로움, 블러 효과
             A.Sharpen(p=1),
             A.Blur(p=1)], p=0.3),
    A.PiecewiseAffine(p=0.3),                               # 어파인 변환
    A.Normalize(),                                          # 정규화 변환
    ToTensorV2()                                            # 텐서로 변환
])

In [ ]:
transform_test = A.Compose([
    A.Resize(450, 650),
    A.Normalize(),
    ToTensorV2()
])

## 데이터셋 및 데이터 로더 생성

In [ ]:
img_dir = '/kaggle/input/plant-pathology-2020-fgvc7/images/'

In [ ]:
dataset_train = ImageDataset(train, img_dir=img_dir, transform=transform_train)
dataset_valid = ImageDataset(train, img_dir=img_dir, transform=transform_test)

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
g = torch.Generator()
g.manual_seed(0)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
batch_size = 4

In [ ]:
loader_train = DataLoader(dataset_train, batch_size=batch_size,
                          shuffle=True, worker_init_fn=seed_worker,
                          generator=g, num_workers=2)

In [ ]:
loader_valid = DataLoader(dataset_valid, batch_size=batch_size,
                          shuffle=False, worker_init_fn=seed_worker,
                          generator=g, num_workers=2)

# 모델 생성

## EfficientNet 모델 생성

In [ ]:
!pip install efficientnet-pytorch==0.7.1

In [ ]:
from efficientnet_pytorch import EfficientNet

In [ ]:
model = EfficientNet.from_pretrained('efficientnet-b7', num_classes=4)
model = model.to(device)

In [ ]:
# # efficientnet-b7의 출력값 개수를 설정하는 또 다른 방법

# # 사전 훈렫된 efficientnet-b7 모델 불러오기
# model = EfficientNet.from_pretrained('efficientnet-b7')

# # 불러온 efficientnet-b7 모델의 마지막 계층 수정
# model._fc = nn.Sequential(
#     nn.Linear(model._fc.in_features, model._fc.out_features), # 2560 -> 1000
#     nn.ReLu(), # 활성화 함수
#     nn.Dropout(p=0.5) # 50% 드롭아웃
#     nn.Linear(model._fc.out_features, 4) # 1000 -> 4
# )

# 모델 훈련 및 성능 검증

## 손실 함수와 옵티마이저 설정

In [ ]:
import torch.nn as nn

In [ ]:
# 손실 함수
criterion = nn.CrossEntropyLoss()

In [ ]:
# 옵티마이저
optimizer = torch.optim .AdamW(model.parameters(), lr=0.00006, weight_decay=0.0001)

## 훈련 및 성능 검증

In [ ]:
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

In [ ]:
epochs = 5

In [ ]:
for epoch in range(epochs):
    # ======================= [훈련] =======================
    model.train()         # 모델을 훈련 상태로 설정
    epoch_train_loss = 0  # 에폭별 손실값 초기화(훈련 데이터용)

    for images, labels in tqdm(loader_train):
        # 이미지, 레이블(타깃값) 데이터 미니배치를 장비에 할당
        images = images.to(device)
        labels = labels.to(device)

        # 옵티마이저 내 기울기 초기화
        optimizer.zero_grad()
        # 순전파: 이미지 데이터를 신경망 모델의 입력값으로 사용해 출력값 계산
        outputs = model(images)
        # 손실 함수를 활용해 outputs와 labels의 손실값 계산
        loss = criterion(outputs, labels)

        # 현재 배치에서의 손실 추가(훈련 데이터용)
        epoch_train_loss += loss.item()
        loss.backward()   #역전파 수행
        optimizer.step()  #가중치 갱신

    # 훈련 데이터 손실값 출력
    print(f'에폭 [{epoch+1}/{epochs}] - 훈련 데이터 손실값: {epoch_train_loss/len(loader_train):.4f}')

    # ======================= [검증] =======================
    model.eval()          # 모델을 평가 상태로 설정
    epoch_valid_loss = 0  # 에폭별 손실값 초기화 (검증 데이터용)
    preds_list = []       # 예측 확률값 저장용 리스트 초기화
    true_onehot_list = [] # 실제 타깃값 저장용 리스트 초기화

    with torch.no_grad(): # 기울기 계산 비활성화
        # 미니배치 단위로 검증
        for images, labels in loader_valid:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            epoch_valid_loss += loss.item()

            preds = torch.softmax(outputs.cpu(), dim=1).numpy() # 예측 확률값
            #실젯값 (원-핫 인코딩 형식)
            true_onehot = torch.eye(4, device=labels.device)[labels].cpu().numpy()

            # 예측 확률값과 실젯값 저장
            preds_list.extend(preds)
            true_onehot_list.extend(true_onehot)

        print(f'에폭 [{epoch+1}/{epochs}] - 검증 데이터 손실값: {epoch_valid_loss/len(loader_valid):.4f}')

In [ ]:
torch.eye(4, device=labels.device)[labels].cpu().numpy()

# 에측 및 결과 제출

In [ ]:
dataset_test = ImageDataset(test, img_dir=img_dir,
                            transform=transform_test, is_test=True)
loader_test = DataLoader(dataset_test, batch_size=batch_size,
                         shuffle=False, worker_init_fn=seed_worker,
                         generator=g, num_workers=2)

## 예측

In [ ]:
model.eval()

In [ ]:
preds = np.zeros((len(test), 4)) # 예측값 저장용 배열 초기화

In [ ]:
with torch.no_grad():
    for i, images in enumerate(loader_test):
        images = images.to(device)
        outputs = model(images)

        # 타깃 예측 확률
        preds_part = torch.softmax(outputs.cpu(), dim=1).squeeze().numpy()
        preds[i*batch_size:(i+1)*batch_size] += preds_part

## 결과 제출

In [ ]:
submission[['healthy', 'multiple_diseases', 'rust', 'scab']] = preds
submission.to_csv('submission.csv', index=False)